In [2]:
pip install pdfplumber pandas xlsxwriter openpyxl

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.6 MB 840.2 kB/s eta 0:00:07
   --- ------------------------------------ 0.5/5.6 MB 840.2 kB/s eta 0:00:07
   ----- ---------------------------------- 0.8/5.6 MB 884.1 kB/s eta 0:00:06
   ------- -------------------------------- 1.0/5.6 MB 932.9 kB/s eta 0:00:05
   --------- ------------------------------ 1.3/5.6 MB 959.4 kB/s eta 0:00:05
   ----------- ---------------------------- 1.6/5.6 MB 987.0 kB/s eta 0:00:05
   ------------- -------------------------- 1.8/5.6 MB 1.0 MB/s eta 0:00:04
   -------------- ------------------------- 2.1/5.6 MB 1.0 MB/s eta 0:00:04
   ---------------- ----------------------- 2.4/5.6 MB 1.1 MB/s eta 0:00:04
   ------------------ --------------------- 2.6/5.6 MB 1.1 MB/s eta 0:00:03
   ------------------

In [1]:
# pip install pdfplumber pandas xlsxwriter openpyxl

import os
import re
import pdfplumber
import pandas as pd

pdf_path  = r"C:/Users/Kevin Byamungu/OneDrive/Documents/GitHub/Projet-tutore/Palmares/2022/Palmares_2022_-HAUT-KATANGA-1_Code-71.pdf"
xlsx_path = r"C:/Users/Kevin Byamungu/OneDrive/Documents/GitHub/Projet-tutore/Palmares/fichiers modifiés/eleves_EQUATEUR12022.xlsx"

# Options conservées
options_map = {
    "101": "LATIN-PHILOSOPHIE",
    "102": "MATHEMATIQUE-PHYSIQUE",
    "103": "CHIMIE-BIOLOGIE",
    "201": "PEDAGOGIE GENERALE",
    "301": "COMMERCIALE ET GESTION",
}
OPTIONS_ALLOWED = set(options_map.keys())

# --- Normalisation du nom d'école ---
def normalize_school_name(name: str) -> str:
    n = (name or "").upper()

    rules = [
        (r'(?<!\w)L\.\s*T\.?(?!\w)', 'LYCEE TECHNIQUE'),     
        (r'(?<!\w)I\.\s*T\.?(?!\w)', 'INSTITUT TECHNIQUE'),  
        (r'(?<!\w)REV\.?(?!\w)', 'REVEREND'),                
        (r'(?<!\w)C[.\s]*S[.\s]?(?!\w)','COMPLEXE SCOLAIRE'),
        (r'(?<!\w)CSFRANCO(?!\w)','COMPLEXE SCOLAIRE FRANCOPHONE'),
        (r'(?<!\w)I\.(?!\w)', 'INSTITUT'),
        (r'(?<!\w)INST(?!\w)', 'INSTITUT'),
        (r'(?<!\w)I.TITUT(?!\w)', 'INSTITUT'),
        (r'(?<!\w)I. SC(?!\w)', 'INSTITUT SCIENTIFIQUE'),
        (r'(?<!\w)GS(?!\w)', 'GROUPE SCOLAIRE'),
        (r'(?<!\w)COL(?!\w)', 'COLLEGE'),
        (r'(?<!\w)COLL\.?(?!\w)', 'COLLEGE'),
        (r'(?<!\w)CM(?!\w)', 'COLLEGE MODERNE'),
        (r'(?<!\w)L\.(?!\w)', 'LYCEE'),
        (r'(?<!\w)L\. M\.(?!\w)', 'LYCEE MADAME'),
        (r'(?<!\w)E\.(?!\w)', 'ECOLE'),
        (r'(?<!\w)ST(?!\w)', 'SAINT'),
        (r'(?<!\w)STE(?!\w)', 'SAINTE'),
        (r'(?<!\w)ND(?!\w)', 'NOTRE DAME'),
        (r'(?<!\w)M\.(?!\w)', 'MARIE'),
        (r'(?<!\w)JEAN-M\.(?!\w)', 'JEAN-MARIE'),
        (r'(?<!\w)COL M\.(?!\w)', 'COLLEGE MARIE'),
        (r'(?<!\w)PST(?!\w)', 'PASTEUR'),
        (r'(?<!\w)PED(?!\w)', 'PEDAGOGIQUE'),
        (r'(?<!\w)D\'EX(?!\w)', 'D\'EXCELLENCE'),
        (r'(?<!\w)PROF(?!\w)', 'PROFESSEUR'),
        (r'(?<!\w)SŒURS(?!\w)', 'SOEURS'),
        (r'(?<!\w)PÈRE(?!\w)', 'PERE'),
    ]
    for pat, rep in rules:
        n = re.sub(pat, rep, n)

    n = re.sub(r'\s+', ' ', n).strip()
    return n

# Détection école
ecole_pattern = re.compile(r'^[A-Z0-9\.\s/\-]+$')
def is_header_like(line: str) -> bool:
    l = line.upper().strip()
    return l.startswith(("OPTION", "PROVINCE", "CODE", "PARTICIP", "RÉUSSIT", "REUSSIT"))

# Regex
percent_re = re.compile(r'^\d+(?:[.,]\d+)?$')
code_ecole_re = re.compile(r'Code\s*:\s*([\d\s/]+)')

def extract_lines(page, bbox):
    words = page.crop(bbox).extract_words(x_tolerance=1, y_tolerance=3)
    if not words:
        return []
    words = sorted(words, key=lambda w: (round(w['top'],1), w['x0']))
    lines, cur_top, buf = [], None, []
    for w in words:
        if cur_top is None or abs(w['top'] - cur_top) <= 5:
            buf.append(w); cur_top = w['top'] if cur_top is None else cur_top
        else:
            lines.append(" ".join(x['text'] for x in sorted(buf, key=lambda t:t['x0'])))
            buf = [w]; cur_top = w['top']
    if buf:
        lines.append(" ".join(x['text'] for x in sorted(buf, key=lambda t:t['x0'])))
    return [ln.strip() for ln in lines if ln.strip()]

def process_column(lines, out_rows):
    ecole_actuelle = None
    code_ecole_actuel = None
    option_actuelle = None
    collecting = False

    for line in lines:
        # nom d'école
        if ecole_pattern.match(line) and not line[0].isdigit() and not is_header_like(line):
            ecole_actuelle = normalize_school_name(line.strip())
            continue

        # code d'école
        m = code_ecole_re.search(line)
        if m:
            raw_code = m.group(1).strip()
            parts = [p.strip() for p in raw_code.split("/") if p.strip()]
            option_actuelle = parts[1][:3] if len(parts) > 1 else None

            code_clean = re.sub(r"[ /]", "", raw_code)
            if option_actuelle and option_actuelle in code_clean:
                code_final = code_clean.replace(option_actuelle, "", 1)
            else:
                code_final = code_clean

            code_ecole_actuel = code_final
            collecting = option_actuelle in OPTIONS_ALLOWED
            continue

        # élève
        tokens = line.split()
        if len(tokens) >= 4 and tokens[0].isdigit():
            if tokens[-2].upper() in ("M","F") and percent_re.match(tokens[-1]):
                if not collecting:
                    continue
                numero = tokens[0].zfill(3)  # <-- ajoute les zéros à gauche
                numero_concat = f"{code_clean}{numero}" if code_clean else numero
                nom, postnom = tokens[1], tokens[2]
                prenom = " ".join(tokens[3:-2])
                sexe, pourcentage = tokens[-2], tokens[-1]
                out_rows.append([
                    numero_concat, nom, postnom, prenom, sexe, pourcentage,
                    ecole_actuelle, code_ecole_actuel or "",
                    option_actuelle, options_map.get(option_actuelle, ""),
                    "KINSHASA-MONT AMBA", "2023"
                ])

# -------- extraction --------
eleves = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        w, h = page.width, page.height
        bboxes = [
            (0, 0, w/2.8, h),          
            (w/2.8, 0, w/1.59, h),     
            (w/1.57, 0, w, h),         
        ]
        for bbox in bboxes:
            lines = extract_lines(page, bbox)
            process_column(lines, eleves)

# -------- écriture Excel --------
headers = [
    "ID Élève", "Nom", "Postnom", "Prénom", "Sexe", "Pourcentage",
    "École", "Code École",
    "Code de l'option", "Option",
    "Province Éducationnelle", "Année"
]

os.makedirs(os.path.dirname(xlsx_path), exist_ok=True)
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    sheet = "Élèves"
    df = pd.DataFrame(eleves, columns=headers)
    df.to_excel(writer, sheet_name=sheet, index=False, header=False, startrow=1)

    workbook  = writer.book
    worksheet = writer.sheets[sheet]
    rows, cols = df.shape
    col_settings = [{"header": h} for h in headers]

    worksheet.add_table(0, 0, rows, cols-1, {
        "columns": col_settings,
        "style":   "Table Style Medium 9",
        "name":    "Eleves",
    })

    for i, h in enumerate(headers):
        worksheet.set_column(i, i, max(12, min(35, len(h)+4)))

print(f"✅ Fichier Excel créé : {xlsx_path} — {len(eleves)} élèves")


✅ Fichier Excel créé : C:/Users/Kevin Byamungu/OneDrive/Documents/GitHub/Projet-tutore/Palmares/fichiers modifiés/eleves_EQUATEUR12022.xlsx — 14800 élèves


In [2]:
# pip install pdfplumber pandas xlsxwriter openpyxl

import os
import re
import pdfplumber
import pandas as pd

# === Chemins d'entrée/sortie ===
pdf_path  = r"C:/Users/Kevin Byamungu/OneDrive/Documents/GitHub/Projet-tutore/Palmares/2022/Palmares_2022_EQUATEUR-1_Code-41.pdf"
xlsx_path = r"C:/Users/Kevin Byamungu/OneDrive/Documents/GitHub/Projet-tutore/Palmares/eleves_EQUATEUR12022.xlsx"

# === Options conservées ===
options_map = {
    "101": "LATIN-PHILOSOPHIE",
    "102": "MATHEMATIQUE-PHYSIQUE",
    "103": "CHIMIE-BIOLOGIE",
    "201": "PEDAGOGIE GENERALE",
    "301": "COMMERCIALE ET GESTION",
}
OPTIONS_ALLOWED = set(options_map.keys())

# --- Normalisation du nom d'école ---
def normalize_school_name(name: str) -> str:
    if not name:
        return ""
    n = name.upper()
    rules = [
        (r'L\.\s*T\.?', 'LYCEE TECHNIQUE'),
        (r'I\.\s*T\.?', 'INSTITUT TECHNIQUE'),
        (r'REV\.?', 'REVEREND'),
        (r'C[.\s]*S[.\s]?', 'COMPLEXE SCOLAIRE'),
        (r'CSFRANCO', 'COMPLEXE SCOLAIRE FRANCOPHONE'),
        (r'I\.', 'INSTITUT'),
        (r'INST', 'INSTITUT'),
        (r'I\.?TITUT', 'INSTITUT'),
        (r'I\.\s*SC', 'INSTITUT SCIENTIFIQUE'),
        (r'GS', 'GROUPE SCOLAIRE'),
        (r'COLL?\.?', 'COLLEGE'),
        (r'CM', 'COLLEGE MODERNE'),
        (r'L\.', 'LYCEE'),
        (r'E\.', 'ECOLE'),
        (r'ST', 'SAINT'),
        (r'STE', 'SAINTE'),
        (r'ND', 'NOTRE DAME'),
        (r"M\.", 'MARIE'),
        (r'JEAN-M\.', 'JEAN-MARIE'),
        (r'COL\s*M\.', 'COLLEGE MARIE'),
        (r'PST', 'PASTEUR'),
        (r'PED', 'PEDAGOGIQUE'),
        (r"D['‘’]?EX", "D'EXCELLENCE"),
        (r'PROF', 'PROFESSEUR'),
        (r'SŒURS', 'SOEURS'),
        (r'PÈRE', 'PERE'),
    ]
    for pat, rep in rules:
        n = re.sub(pat, rep, n)
    n = re.sub(r'\s+', ' ', n).strip()
    return n

# --- Détection école ---
ecole_pattern = re.compile(r"^[A-Za-zÀ-ÿ0-9\s'‘’\.\-_/]+$")
def is_header_like(line: str) -> bool:
    l = line.upper().strip()
    return l.startswith(("OPTION", "PROVINCE", "CODE", "PARTICIP", "RÉUSSIT", "REUSSIT"))

# --- Regex utiles ---
percent_re = re.compile(r'^\d+(?:[.,]\d+)?$')
code_ecole_re = re.compile(r'Code\s*:\s*([\d\s/]+)')

# --- Extraction lignes du PDF ---
def extract_lines(page, bbox):
    words = page.crop(bbox).extract_words(x_tolerance=1, y_tolerance=3)
    if not words:
        return []
    words = sorted(words, key=lambda w: (round(w['top'], 1), w['x0']))
    lines, cur_top, buf = [], None, []
    for w in words:
        if cur_top is None or abs(w['top'] - cur_top) <= 5:
            buf.append(w)
            cur_top = w['top'] if cur_top is None else cur_top
        else:
            lines.append(" ".join(x['text'] for x in sorted(buf, key=lambda t: t['x0'])))
            buf = [w]
            cur_top = w['top']
    if buf:
        lines.append(" ".join(x['text'] for x in sorted(buf, key=lambda t: t['x0'])))
    return [ln.strip() for ln in lines if ln.strip()]

# --- Traitement d'une colonne ---
def process_column(lines, out_rows):
    ecole_actuelle = None
    code_ecole_actuel = None
    option_actuelle = None
    collecting = False
    code_clean = None

    for line in lines:
        # Nom d'école
        if ecole_pattern.match(line) and not line[0].isdigit() and not is_header_like(line):
            ecole_actuelle = normalize_school_name(line.strip())
            continue

        # Code d'école
        m = code_ecole_re.search(line)
        if m:
            raw_code = m.group(1).strip()
            parts = [p.strip() for p in raw_code.split("/") if p.strip()]
            option_actuelle = parts[1][:3] if len(parts) > 1 else None

            # Vérifie si option autorisée
            if option_actuelle not in OPTIONS_ALLOWED:
                collecting = False
                continue

            code_clean = re.sub(r"[ /]", "", raw_code)
            if option_actuelle and option_actuelle in code_clean:
                code_final = code_clean.replace(option_actuelle, "", 1)
            else:
                code_final = code_clean

            code_ecole_actuel = code_final
            collecting = True
            continue

        # Élève
        tokens = line.split()
        if len(tokens) >= 4 and tokens[0].isdigit():
            if tokens[-2].upper() in ("M", "F") and percent_re.match(tokens[-1]):
                if not collecting:
                    continue
                numero = tokens[0].zfill(3)
                numero_concat = f"{code_clean}{numero}" if code_clean else numero
                nom, postnom = tokens[1], tokens[2]
                prenom = " ".join(tokens[3:-2])
                sexe, pourcentage = tokens[-2], tokens[-1]
                out_rows.append([
                    numero_concat, nom, postnom, prenom, sexe, pourcentage,
                    ecole_actuelle, code_ecole_actuel or "",
                    option_actuelle, options_map.get(option_actuelle, ""),
                    "HAUT-KATANGA 1", "2022"
                ])

# --- Extraction principale ---
eleves = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        w, h = page.width, page.height
        bboxes = [
            (0, 0, w/2.8, h),
            (w/2.8, 0, w/1.59, h),
            (w/1.57, 0, w, h),
        ]
        for bbox in bboxes:
            lines = extract_lines(page, bbox)
            process_column(lines, eleves)

# --- Écriture Excel ---
headers = [
    "ID Élève", "Nom", "Postnom", "Prénom", "Sexe", "Pourcentage",
    "École", "Code École",
    "Code de l'option", "Option",
    "Province Éducationnelle", "Année"
]

os.makedirs(os.path.dirname(xlsx_path), exist_ok=True)
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    df = pd.DataFrame(eleves, columns=headers)
    df.to_excel(writer, sheet_name="Élèves", index=False, header=False, startrow=1)

    workbook = writer.book
    worksheet = writer.sheets["Élèves"]
    rows, cols = df.shape
    col_settings = [{"header": h} for h in headers]

    worksheet.add_table(0, 0, rows, cols - 1, {
        "columns": col_settings,
        "style": "Table Style Medium 9",
        "name": "Eleves",
    })

    for i, h in enumerate(headers):
        worksheet.set_column(i, i, max(12, min(35, len(h) + 4)))

print(f"✅ Fichier Excel créé : {xlsx_path} — {len(eleves)} élèves extraits.")


✅ Fichier Excel créé : C:/Users/Kevin Byamungu/OneDrive/Documents/GitHub/Projet-tutore/Palmares/eleves_EQUATEUR12022.xlsx — 4400 élèves extraits.
